In [2]:
import requests
import json
import time
import os
import random

# Configuration
URL = "http://localhost:11434/api/generate"
# Use a raw string (r"") for Windows paths to avoid backslash errors
SAVE_PATH = r"C:\datasets\llm-data\sft\LLama_synthetic_data_1.json"

# Expanded Topic Bank
topic_bank = {
    "Statistics_Descriptive": [
        "Measures of Central Tendency: Robustness of Mean vs Median vs Mode",
        "Measures of Dispersion: Variance, Standard Deviation, and Coefficient of Variation",
        "Skewness and Kurtosis: Interpreting distribution shapes",
        "Percentiles, Quartiles, and Interquartile Range (IQR) for outlier detection",
        "Box-and-Whisker plot construction and interpretation",
        "Z-score normalization and Empirical Rule (68-95-99.7)",
        "Covariance vs Correlation: Mathematical foundations",
        "Frequency distributions and Histogram binning strategies",
        "Cumulative Distribution Functions (CDF) vs Probability Density Functions (PDF)",
        "Standard Error of the Mean vs Standard Deviation"
    ],
    "Statistics_Inferential": [
        "Hypothesis Testing: Null vs Alternative and P-value interpretation",
        "Type I and Type II errors: The Power of a statistical test",
        "Confidence Intervals: Construction and Margin of Error",
        "T-tests: Independent, Paired, and One-sample implementations",
        "ANOVA (Analysis of Variance): F-statistic and Post-hoc testing (Tukey)",
        "Chi-Square Tests: Goodness of Fit and Independence",
        "Non-parametric tests: Mann-Whitney U and Wilcoxon Signed-Rank",
        "Bayesian Inference: Prior, Likelihood, and Posterior calculations",
        "Maximum Likelihood Estimation (MLE) for parameter tuning",
        "Bootstrap Resampling and Jackknife methods for inference",
        "Degrees of Freedom: Mathematical necessity in small samples",
        "Multiple Testing Problem and Bonferroni Correction"
    ],
    "Data_Science": [
        "Exploratory Data Analysis (EDA) best practices",
        "Handling missing data: Imputation techniques",
        "Feature Engineering for categorical variables",
        "A/B Testing design and metrics",
        "Dimensionality Reduction using PCA and t-SNE",
        "Data Leakage detection and prevention",
        "Time Series Decomposition (Trend, Seasonality, Residuals)",
        "Evaluating model performance with ROC-AUC and F1-score",
        "Data normalization vs standardization",
        "Addressing Selection Bias in datasets"
    ],
    "Machine_Learning": [
        "Bias-Variance Tradeoff in complex models",
        "Regularization techniques: Lasso (L1) vs Ridge (L2)",
        "Ensemble methods: Bagging vs Boosting",
        "Gradient Boosting Machines (XGBoost, LightGBM)",
        "Support Vector Machines and the Kernel Trick",
        "Random Forest feature importance interpretation",
        "Hyperparameter tuning using Bayesian Optimization",
        "K-Nearest Neighbors for classification and regression",
        "Stochastic Gradient Descent (SGD) mechanics",
        "Cross-Validation strategies for small datasets"
    ],
    "Deep_Learning": [
        "Backpropagation and the Chain Rule",
        "Vanishing and Exploding Gradient problems",
        "CNN Architectures: ResNet, VGG, and Inception",
        "Recurrent Neural Networks (RNN) and LSTM/GRU units",
        "Dropout and Batch Normalization for regularization",
        "Weight Initialization: Xavier vs He methods",
        "Transfer Learning using pre-trained models",
        "Generative Adversarial Networks (GANs) basics",
        "Autoencoders for anomaly detection",
        "Activation Functions: ReLU, Leaky ReLU, and GELU"
    ]
}

def clean_json_string(text):
    """Removes potential markdown block wrappers."""
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[-1]
        if text.endswith("```"):
            text = text.rsplit("```", 1)[0]
    return text.strip()

def generate_sample(topic):
    prompt = f"""
Generate ONE high-quality data science training sample.
Topic: {topic}

STRICT RULES:
- Return valid JSON matching the FORMAT below.
- Do NOT include any text, headers, or markdown outside the JSON object.
- Explanation must be intuitive (3–5 sentences).
- Code must be complete, executable Python code.

FORMAT:
{{
  "instruction": "clear question about {topic}",
  "input": "",
  "output": "### Explanation:\\n[Your explanation here]\\n\\n### Code:\\n[Your python code here]"
}}
"""
    try:
        res = requests.post(URL, json={
            "model": "llama3",
            "prompt": prompt,
            "stream": False,
            "format": "json",
            "options": {
                "temperature": 0.8,
                "num_predict": 2048
            }
        }, timeout=120)
        
        res.raise_for_status()
        raw_response = res.json().get("response", "")
        cleaned_text = clean_json_string(raw_response)
        return json.loads(cleaned_text)
    except Exception as e:
        print(f" Error: {e}")
        return None

# Ensure directory exists
os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)

dataset = []
target_count = 200

# Flatten topics for easy iteration
all_topics = [t for sublist in topic_bank.values() for t in sublist]

print(f"Starting generation of {target_count} samples...")

for i in range(target_count):
    topic = random.choice(all_topics)
    print(f"[{i+1}/{target_count}] Topic: {topic}...", end=" ")

    sample = generate_sample(topic)

    if sample and "output" in sample:
        out = sample["output"]
        if "### Explanation" in out and "### Code" in out and len(out) > 150:
            dataset.append(sample)
            print(f"Saved!")
            # Save progress incrementally
            if (i + 1) % 5 == 0:
                with open(SAVE_PATH, "w") as f:
                    json.dump(dataset, f, indent=2)
        else:
            print("Quality check failed.")
    else:
        print("Generation failed.")

    time.sleep(1)

# Final Save
with open(SAVE_PATH, "w") as f:
    json.dump(dataset, f, indent=2)

print(f"\nDONE: {len(dataset)} samples saved to {SAVE_PATH}")

Starting generation of 200 samples...
[1/200] Topic: Gradient Boosting Machines (XGBoost, LightGBM)... Saved!
[2/200] Topic: Handling missing data: Imputation techniques... Saved!
[3/200] Topic: A/B Testing design and metrics... Saved!
[4/200] Topic: Ensemble methods: Bagging vs Boosting... Saved!
[5/200] Topic: Hyperparameter tuning using Bayesian Optimization... Saved!
[6/200] Topic: Vanishing and Exploding Gradient problems... Saved!
[7/200] Topic: Generative Adversarial Networks (GANs) basics... Saved!
[8/200] Topic: Ensemble methods: Bagging vs Boosting... Saved!
[9/200] Topic: Confidence Intervals: Construction and Margin of Error... Saved!
[10/200] Topic: Standard Error of the Mean vs Standard Deviation... Saved!
[11/200] Topic: Z-score normalization and Empirical Rule (68-95-99.7)... Saved!
[12/200] Topic: T-tests: Independent, Paired, and One-sample implementations... Saved!
[13/200] Topic: Z-score normalization and Empirical Rule (68-95-99.7)... Saved!
[14/200] Topic: Transfe